# Week 60, Day 3: Decorators II

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w060d3.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
import functools

def repeat(times):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            return [func(*args, **kwargs) for _ in range(times)]
        return wrapper
    return decorator

@repeat(3)
def roll():
    return "six"

print(roll(), roll.__name__)
print(repeat(2)(lambda: "x")())

In [ ]:
import functools

def retry(times, exceptions=(Exception,)):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            for attempt in range(1, times + 1):
                try:
                    return func(*args, **kwargs)
                except exceptions as e:
                    print(f"attempt {attempt} failed: {e}")
                    if attempt == times:
                        raise
        return wrapper
    return decorator

replies = iter(["timeout", "timeout", "ok"])

@retry(times=3, exceptions=(ConnectionError,))
def fetch():
    reply = next(replies)
    if reply != "ok":
        raise ConnectionError(reply)
    return reply

print(fetch())

In [ ]:
import functools

def logged(func=None, *, prefix="call"):
    if func is None:
        return lambda f: logged(f, prefix=prefix)
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        print(prefix, func.__name__, args)
        return func(*args, **kwargs)
    return wrapper

@logged
def a(x):
    return x

@logged(prefix=">>")
def b(x):
    return x

a(1)
b(2)

In [ ]:
import functools

class CountCalls:
    def __init__(self, func):
        functools.update_wrapper(self, func)
        self.func = func
        self.calls = 0

    def __call__(self, *args, **kwargs):
        self.calls += 1
        return self.func(*args, **kwargs)

    def reset(self):
        self.calls = 0

@CountCalls
def ping():
    """Say pong."""
    return "pong"

ping()
ping()
print(ping.calls, ping.__name__, ping.__doc__)
ping.reset()
print(ping.calls)

In [ ]:
import functools

def logged_method(method):
    @functools.wraps(method)
    def wrapper(self, *args, **kwargs):
        print(f"{type(self).__name__}.{method.__name__}{args}")
        return method(self, *args, **kwargs)
    return wrapper

def with_repr(cls):
    def __repr__(self):
        fields = ", ".join(f"{k}={v!r}" for k, v in vars(self).items())
        return f"{cls.__name__}({fields})"
    cls.__repr__ = __repr__
    return cls

@with_repr
class Account:
    def __init__(self, owner, balance=0):
        self.owner, self.balance = owner, balance

    @logged_method
    def deposit(self, amount):
        self.balance += amount

acc = Account("Ada")
acc.deposit(50)
print(acc)

In [ ]:
class Temperature:
    def __init__(self, celsius):
        self._celsius = celsius

    def _get(self):
        return self._celsius

    def _set(self, value):
        if value < -273.15:
            raise ValueError("below absolute zero")
        self._celsius = value

    celsius = property(_get, _set)            # what @property and @celsius.setter build

    @property
    def fahrenheit(self):
        return self._celsius * 9 / 5 + 32

t = Temperature(20)
t.celsius = 100
print(t.celsius, t.fahrenheit, type(Temperature.__dict__["celsius"]).__name__)

> **Key idea.** A decorator factory takes settings and returns a decorator: three nested functions, so @retry(3) means f = retry(3)(f). A decorator can work with or without brackets by taking the function as an optional first parameter and the settings as keyword-only. A class can be a decorator: __init__ receives the function (copy its details with functools.update_wrapper) and __call__ runs at each call, but prefer functions for methods. Function decorators work on methods, with self passed through. Class decorators receive and return a class, as @dataclass and @total_ordering do. property, classmethod and staticmethod are classes; @x.setter returns a new property, so the setter must share the getter's name.

### Exercise: A decorator with a setting

Write `repeat(times)`: a decorator factory whose wrapper calls the function `times` times and returns a list of the results, and `limit_length(n)`: a factory whose wrapper keeps the (string) result if it's at most n characters long, and otherwise returns its first n characters followed by `"..."`. Both with `functools.wraps`.

In [ ]:
import functools

def repeat(times):
    ___

def limit_length(n):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_times():\n    \"decorator factories\"\n    @repeat(3)\n    def hi():\n        return \"hi\"\n    assert hi() == [\"hi\", \"hi\", \"hi\"] and hi.__name__ == \"hi\"\n    @limit_length(5)\n    def say(text):\n        return text\n    assert say(\"hello\") == \"hello\" and say(\"hello world\") == \"hello...\" and say.__name__ == \"say\"\n    assert repeat(0)(lambda: 1)() == []")

### Exercise: Trying again

Write `retry(times, exceptions=(Exception,), log=None)`: a decorator factory whose wrapper calls the function up to `times` times while it raises one of `exceptions`, re-raising the last exception if every attempt fails. Exceptions **not** in `exceptions` pass straight through. If `log` is a list, append `(func.__name__, attempt, repr(error))` for each failed attempt.

In [ ]:
import functools

def retry(times, exceptions=(Exception,), log=None):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_retry():\n    \"a factory with state\"\n    attempts = []\n    replies = iter([ValueError(\"a\"), ValueError(\"b\"), \"ok\"])\n    @retry(3, exceptions=(ValueError,), log=attempts)\n    def flaky():\n        r = next(replies)\n        if isinstance(r, Exception):\n            raise r\n        return r\n    assert flaky() == \"ok\" and attempts == [(\"flaky\", 1, \"ValueError('a')\"), (\"flaky\", 2, \"ValueError('b')\")]\n    @retry(2)\n    def broken():\n        raise KeyError(\"x\")\n    try:\n        broken()\n        assert False\n    except KeyError:\n        pass\n    @retry(5, exceptions=(ValueError,))\n    def wrong_kind():\n        raise TypeError(\"no\")\n    try:\n        wrong_kind()\n        assert False\n    except TypeError:\n        pass\n    assert flaky.__name__ == \"flaky\"")

### Exercise: Brackets optional

Write `tagged(func=None, *, name="b")`: a decorator usable as `@tagged` (wrapping the string result in `<b>...</b>`) or `@tagged(name="i")` (using that tag instead).

In [ ]:
import functools

def tagged(func=None, *, name="b"):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_optional():\n    \"both forms\"\n    @tagged\n    def a():\n        return \"bold\"\n    @tagged(name=\"em\")\n    def b():\n        return \"stress\"\n    assert a() == \"<b>bold</b>\" and b() == \"<em>stress</em>\" and a.__name__ == \"a\" and b.__name__ == \"b\"\n    assert tagged(lambda: \"x\")() == \"<b>x</b>\"")

### Exercise: A decorator class with memory

Write a class `Memo` used as a decorator: it caches results by positional arguments, counts `hits` and `misses`, has `clear()` (empties the cache and resets both counts) and `info()` (a tuple `(hits, misses, size)`), and keeps the function's name and docstring with `functools.update_wrapper`.

In [ ]:
import functools

class Memo:
    def __init__(self, func):
        ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_memo():\n    \"a class-based decorator\"\n    @Memo\n    def square(x):\n        \"\"\"Square it.\"\"\"\n        return x * x\n    assert square(3) == 9 and square(3) == 9 and square(4) == 16\n    assert square.info() == (1, 2, 2) and square.__name__ == \"square\" and square.__doc__ == \"Square it.\"\n    square.clear()\n    assert square.info() == (0, 0, 0)\n    @Memo\n    def fib(n):\n        return n if n < 2 else fib(n - 1) + fib(n - 2)\n    assert fib(80) == 23416728348467685 and fib.info()[1] == 81")

### Exercise: Decorating classes

Write class decorators: `registry` is a dict, and `register(cls)` adds the class under its name and returns it; `frozen(cls)` makes instances read-only **after** `__init__` finishes, by wrapping `__init__` to set a flag and adding a `__setattr__` that raises `AttributeError(f"{cls.__name__} is frozen")` once the flag is set. (Use `object.__setattr__` inside, as in Week 31.)

In [ ]:
import functools

registry = {}

def register(cls):
    ___

def frozen(cls):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_classes():\n    \"class decorators\"\n    @register\n    @frozen\n    class Point:\n        def __init__(self, x, y):\n            self.x, self.y = x, y\n    p = Point(1, 2)\n    assert (p.x, p.y) == (1, 2) and registry[\"Point\"] is Point\n    try:\n        p.x = 5\n        assert False\n    except AttributeError as e:\n        assert str(e) == \"Point is frozen\"\n    @frozen\n    class Box:\n        def __init__(self, items):\n            self.items = list(items)\n    b = Box([1])\n    b.items.append(2)\n    assert b.items == [1, 2] and Box.__init__.__name__ == \"__init__\"")

### Bug hunt: A factory used without its brackets

`repeat` is a factory, so it must be **called** with its setting: `@repeat(2)`. Written as `@repeat`, the function itself becomes `times`, and calling the result fails with `TypeError`. Add the brackets and setting so it prints `['ok', 'ok']`.

In [ ]:
import functools

def repeat(times):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args):
            return [func(*args) for _ in range(times)]
        return wrapper
    return decorator

@repeat
def check():
    return "ok"

try:
    print(check())
except TypeError:
    print("TypeError")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"call the factory\"\n    expect_output(\"['ok', 'ok']\")")

### Bug hunt: A setter with a different name

The setter must have the **same name** as the property. Named `set_price`, it makes a second property called `set_price`, and `price` stays read-only, so the assignment raises `AttributeError`. Rename the setter `price` so it prints `12.5`.

In [ ]:
class Item:
    def __init__(self, price):
        self._price = price

    @property
    def price(self):
        return self._price

    @price.setter
    def set_price(self, value):
        self._price = round(value, 2)

item = Item(10)
try:
    item.price = 12.499
    print(item.price)
except AttributeError:
    print("AttributeError")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"the same name\"\n    expect_output(\"12.5\")")

### Bug hunt: A method decorator that drops self

The wrapper takes exactly one argument and passes it as the method's first, so `self` is lost: calling `deposit(50)` passes `self` as `x` and nothing as `amount`, raising `TypeError`. Accept and pass on `*args, **kwargs` so it prints `50`.

In [ ]:
import functools

def checked(method):
    @functools.wraps(method)
    def wrapper(x):
        return method(x)
    return wrapper

class Account:
    def __init__(self):
        self.balance = 0

    @checked
    def deposit(self, amount):
        self.balance += amount
        return self.balance

try:
    print(Account().deposit(50))
except TypeError:
    print("TypeError")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"pass self along\"\n    expect_output(\"50\")")

### Puzzle: Three levels

Arrange the lines of a decorator factory that adds a fixed prefix to a function's string result. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # @retry(3) above def f means f = ...: as a string like "retry(3)(f)"
q = ___   # type(Temperature.__dict__["fahrenheit"]).__name__ for today's Temperature class
r = ___   # True or False: a class decorator receives an instance
s = ___   # what @x.setter returns: "a new property" or "the setter function"
t = ___   # the function form of functools.wraps used in a class decorator's __init__, as a string

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == \"retry(3)(f)\" and q == \"property\" and r is False, \"p, q and r\"\n    assert s == \"a new property\" and t == \"update_wrapper\", \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: time-aware decorators

Two decorator factories whose behaviour depends on time. To make them testable, both take a `clock` argument: a function returning the current time in seconds (default `time.monotonic`).

- `rate_limited(calls, period, clock=time.monotonic)`: the wrapped function may run at most `calls` times in any `period` seconds (a sliding window, as in Week 56's rate limiter); a call over the limit raises `RuntimeError(f"{name} called too often")` without running the function.
- `expiring_cache(seconds, clock=time.monotonic)`: caches results by positional arguments, but a cached result older than `seconds` is recomputed. The wrapper has `cache_info()` returning `(hits, misses)` and `cache_clear()`.

Both must keep the function's name with `functools.wraps`.

In [ ]:
import functools, time
from collections import deque

def rate_limited(calls, period, clock=time.monotonic):
    ___

def expiring_cache(seconds, clock=time.monotonic):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_time():\n    \"factories with an injectable clock\"\n    now = [0.0]\n    clock = lambda: now[0]\n    @rate_limited(2, 10, clock=clock)\n    def ping():\n        return \"pong\"\n    assert ping() == \"pong\" and ping() == \"pong\"\n    try:\n        ping()\n        assert False\n    except RuntimeError as e:\n        assert str(e) == \"ping called too often\"\n    now[0] = 10.5\n    assert ping() == \"pong\" and ping.__name__ == \"ping\"\n    runs = []\n    @expiring_cache(60, clock=clock)\n    def rate(currency):\n        runs.append(currency)\n        return len(runs)\n    now[0] = 100\n    assert rate(\"EUR\") == 1 and rate(\"EUR\") == 1 and rate(\"USD\") == 2\n    now[0] = 159\n    assert rate(\"EUR\") == 1\n    now[0] = 161\n    assert rate(\"EUR\") == 3 and rate.cache_info() == (2, 3) and rate.__name__ == \"rate\"\n    rate.cache_clear()\n    assert rate.cache_info() == (0, 0) and rate(\"USD\") == 4")

## Warm-up

`@repeat(3)` first **calls** `repeat(3)`, which returns a decorator; that decorator is then applied to `roll`. Three functions, nested.

## Decorators with settings

A decorator receives exactly one thing: the function. To give it settings, add a level: a **decorator factory** is a function that takes the settings and **returns a decorator**:

```python
def retry(times):              # level 1: the settings
    def decorator(func):       # level 2: the decorator itself
        @functools.wraps(func)
        def wrapper(*args, **kwargs):   # level 3: what runs at each call
            ...
        return wrapper
    return decorator
```

`@retry(times=3)` means `func = retry(times=3)(func)`. Each level is a closure over the ones outside it, so the wrapper can use `times` and `func`. `functools.wraps(func)` is itself a factory of exactly this kind: `wraps(func)` returns the decorator you put on the wrapper.

## With or without brackets

`@retry(3)` and `@retry` mean different things: the second passes the **function** as `times`. A decorator can accept both forms by checking what it was given: if its first argument is a function (and no settings were given), decorate it at once; otherwise return a decorator. The usual pattern makes the function an optional first parameter and the settings keyword-only:

```python
def logged(func=None, *, prefix="call"):
    if func is None:                       # used as @logged(prefix=...)
        return lambda f: logged(f, prefix=prefix)
    ...                                    # used as @logged
```

`functools.lru_cache` works this way: `@lru_cache` and `@lru_cache(maxsize=32)` are both allowed.

## Decorators as classes

Any **callable** can be a decorator, including a class: `@CountCalls` calls `CountCalls(func)`, making an instance, and the instance replaces the function, so it needs `__call__` (Week 31). State lives in ordinary attributes. Use `functools.update_wrapper(self, func)` in `__init__` (the function form of `wraps`) to copy the name and docstring. Class-based decorators read well when the decorator has a lot of state or several methods (`.reset()`, `.stats()`).

One catch: an instance of such a class doesn't behave like a function when put on a **method** (it isn't bound to the instance automatically), so prefer function decorators for methods.

## Methods and classes

**Methods**: a function decorator works on methods unchanged, because a method is a function whose first argument is `self`. With `*args`, the wrapper passes `self` along without knowing; if it needs the instance, it can take `self` explicitly: `def wrapper(self, *args, **kwargs)`.

**Class decorators**: `@decorator` above `class` passes the **class** to the decorator, which may change it (add a method, check its attributes) or register it, and returns a class. `@dataclass` (Week 32) and `@total_ordering` (Week 31) are class decorators: `@dataclass` reads the annotations and writes `__init__`, `__repr__` and `__eq__` onto the class.

## How property works

`property` is a built-in **class**. `property(fget, fset)` makes an object that, stored as a class attribute, turns attribute access into calls: reading `obj.name` calls `fget(obj)`, assigning `obj.name = v` calls `fset(obj, v)`. (The mechanism that lets a class attribute intercept access like this is a **descriptor**, coming later in Phase 5.)

Used as a decorator, `@property` above `def celsius(self)` means `celsius = property(celsius)`: a property with only a getter. Then `@celsius.setter` above a second `def celsius(self, value)` calls the property's `setter` method, which returns a **new** property with the same getter and this setter, and binds it to the same name. That's why the setter must have **the same name** as the getter.

`@classmethod` and `@staticmethod` are classes too: they wrap a function so that, looked up through the class, it receives the class (or nothing) instead of an instance.

## Practice

## Quiz

## Daily challenge

## Before you go

Write, from memory, the three levels of a decorator factory, and say in one sentence what each level receives. Tomorrow: the `functools` module, including the real `cache`, `lru_cache`, `partial` and `singledispatch`.